In [ ]:
# %%capture
# !pip install langchain>=0.1.17 openai>=1.13.3 langchain_openai>=0.1.6 transformers>=4.40.1 datasets>=2.18.0 accelerate>=0.27.2 sentence-transformers>=2.5.1 duckduckgo-search>=5.2.2 langchain_community
# !CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python==0.2.69

# 零、总述

本章中，我们来进一步提升使用 LLM 的体验以及他的输出质量，我们将探索以下几种方法：

- Model I/O: 用 LLMs 加载和工作
- Memory: 帮助 LLM 记忆
- Agents: 使用外部工具组合复杂行为
- Chains: 连接方法和模块

这些方法都集成在 LangChain 中，它是一个用于使用 LLM 的完整框架，它包含多个模块化组件，这些组件可以通过链连接在一起，从而构建复杂的 LLM 系统，如下图所示

<center>
<img src="./resources/langchain_intro.png">
</center>

# 一、Model I/O: 使用 LangChain 加载量化模型

我们使用 LangChain 来扩展 LLM 的能力之前，首先需要加载我们的 LLM，我们将继续使用 `Phi-3` 模型，不过这一次我们使用它的 GGUF 版本，GGUF 模型是原始模型经过一种称为量化（quantization）的方法压缩后得到的版本。量化通过减少表示 LLM 参数所需的比特数来降低模型的存储和运行成本。

比特（bit）由一系列 0 和 1 组成，通过二进制形式对数值进行编码，使用更多的比特可以表示更广泛的数值范围，但同时也需要更多的内存来存储这些数值，如下图所示

<img src="./resources/different_precision.png">

量化会减少 LLM 参数所需的比特数，同时尽可能保留原始信息。这会带来一定程度的精度损失，但通常是值得的，因为量化后的模型运行速度更快、所需 VRAM 更少，而且其准确性通常与原始模型非常接近

> 这里分享一个经验法则

**建议使用至少 4-bit 的量化模型，这些模型在压缩与准确率中达到了很好的平衡。** 即使使用 3-bit 或者甚至更小的 2-bit 的模型，但它们的表现会明显降级，这样你反而可以使用高精度的更小的模型来达到相同的效果

我们使用 FP16 这种变体，代表 16-bit

In [ ]:
from tempfile import template
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

我们使用 `Llama-cpp-python` 来加载模型

In [ ]:
from langchain import LlamaCpp

llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

In [ ]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

不幸的是，没有得到任何输出，这是因为 `Phi-3` 需要一个特定的提示词模板。我们下面可以使用 LangChain 来构建这个提示词模板（chains），而不是每次使用的时候都复制一遍。

# 二、Chains: 扩展 LLM 的能力

LangChain 得名于 chains。虽然我们独立可以独立运行 LLM，但是，当外部组件使用它们时，甚至当它们彼此连接时，chains 仍然能展示出强大的能力。Chains 不仅能扩展 LLM 的能力，而且多种链之间还可以彼此连接。

## 2.1 在 Chain 中的单个链接：提示词模板

首先，我们从创建一条链开始：命名 `Phi-3` 预期的提示词模板。在之前的章节中，我们使用的是 `transformers.pipeline` 自动的应用提示词的模板，但是如果分布在不同包中，这些提示词模板可能会被重复定义。所以，我们现在使用 LangChain，下面是一个很好的例子。

我们的思想就是：将提示词模版和 LLM 连接起来，从而得到我们预期的输出。而不是每次使用 LLM 的时候去复制粘贴它。

`Phi-3` 模型的提示词模板由四个主要部分构成：

- `<s>`：指明提示词的开始位置
- `<|user|>`：指定用户提示词的开始
- `<|assistant|>`：指定模型输出的开始
- `<|end|>`：指定提示词的结束位置或者模型输出的结束位置

我们使用 `PromptTemplate` 来创建 LLM 的模版

In [ ]:
from langchain import PromptTemplate

template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

我们把 prompt 和 llm 链在一起

In [ ]:
basic_chain = prompt | llm

通过 `invoke` 来使用这个链

In [ ]:
basic_chain.invoke(
    {
        "input_prompt": "Hi!My name is Maarten. What is 1 + 1?",
    }
)

可以看到，它输出了不错的结果

## 2.2 用多个提示词组装成链

在之前的例子中，我们创建了一个非常单一的链，因为这个例子相当直接，没有任何可中间处理的步骤，但是，大多数情况下，应用都会涉及比较长且复杂的提示词来生成响应。我们可以将复杂的提示词分成更小的可序列运行的子任务，但这需要多次调用 LLM。如下图所示

<center>
<img src="./resources/sequential_chains.png">
</center>


举个例子，考虑生成故事的情况。我们可以使用提示词询问 LLM 让其生成故事的细节，如标题，摘要，角色描述等等。现在，我们不是要把所有信息都放在单个提示词中，而是将此提示词分成更小的可管理的子任务。

假设我们要生成的故事有以下三个组件：

- 标题
- 主角的描述
- 故事摘要

我们按照下图展示的链来生成故事

<center>
<img src="./resources/generate_story.png">
</center>

为了生成故事情节，我们使用 LangChain 来描述第一个组件，叫标题，这个组件仅需要用户的输入即可。我们定义模版，使用 `"summary"` 变量作为输入，`"title"` 作为输出。即："Create a title for a story about {summary}"

In [ ]:
from langchain import LLMChain

template = """
<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>
"""

title_prompt = PromptTemplate(
    template=template,
    input_variables=["summary"]
)

title = LLMChain(
    llm=llm,
    prompt=title_prompt,
    output_key="title"
)

In [ ]:
title.invoke({"summary": "a girl that lost her mother"})

注意到，我们在输出结果中看到了输入(`"summary"`)和输出(`"title"`)

让我们再来生成下一个组件，命名为角色。我们使用 summary 和 之前生成的 title，为了确保使用了前面的组件，我们的提示词中要带 `{summary}` 和 `{title}` 标签

In [ ]:
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""

character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

虽然我们可以先生成角色之后再生成对应的描述，但我们不那么做

让我们创建最后的组件，我们会使用 summary, title, 和角色描述来生成故事

In [ ]:
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main character is: {character}. Only return the story and it cannot be longer than one paragraph. <|end|>
<|assistant|>"""

story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

现在我们将它们连接起来

In [ ]:
llm_chain = title | character | story

In [ ]:
llm_chain.invoke("a girl that lost her mother")

可以看到，我们通过精简的提示词就可以生成故事。将这些问题分成更小的任务的另一个优点就是我们可以独立访问各个组件。

# 三、Memory：帮助 LLM 记住对话

## 3.1 Conversation Buffer: 对话的缓冲区

## 3.2 Windowed Conversation Buffer: 窗口化的对话缓冲区

## 3.3 Conversation Summary: 对话摘要

# 四、Agents: 创建有 LLM 的系统

## 4.1 Agents 之后的驱动力：一步步推理

## 4.2 LangChain 中的 ReAct